Hugging Face transformers 라이브러리를 사용하여 문서 요약 모델을 구현하는 미션입니다. 데이터 로드 및 전처리부터 요약 모델 실행, 결과 평가까지 전체 파이프라인을 구축해 보세요.

In [2]:
import os

ROOT_DIR = os.getcwd()

# 코랩 모드
if ROOT_DIR == "/content":
    pass
    # print("[[ colab ]]")
    
    # import unicodedata
    
    # DATA_DIR = os.path.join(ROOT_DIR, "data")
    # TEXT_DIR = os.path.join(ROOT_DIR, "raw")

    # if "raw.tar.gz" not in os.listdir():
    #     # subprocess()
    #     !wget https://github.com/wonbywondev/ML-DL/releases/download/data-v3/raw.tar.gz
    # else:
    #     print("· raw.tar.gz (O)")


    # if not os.path.exists(TEXT_DIR):
    #     # !tar -xzvf raw.tar.gz -C /content
    # else:
    #     print("· data (O)")


    # if not os.path.exists(DATA_DIR):
    #     os.mkdir(DATA_DIR)


    # train_json_path = os.path.join(TEXT_DIR, "일상생활및구어체_한영_train_set.json")
    # val_json_path = os.path.join(TEXT_DIR, "일상생활및구어체_한영_valid_set.json")

    # train_json_path = unicodedata.normalize("NFC", train_json_path)
    # val_json_path = unicodedata.normalize("NFC", val_json_path)

# 로컬 모드
else:
    print("[[ local ]]")
    ROOT_DIR = "/".join(ROOT_DIR.split("/")[:-1])
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    RAW_DIR = os.path.join(DATA_DIR, "raw")

    train_edit_json_path = os.path.join(RAW_DIR, "train_original_editorial.json")
    train_law_json_path = os.path.join(RAW_DIR, "train_original_news.json")
    train_news_json_path = os.path.join(RAW_DIR, "train_original_law.json")
    val_edit_json_path = os.path.join(RAW_DIR, "valid_original_editorial.json")
    val_law_json_path = os.path.join(RAW_DIR, "valid_original_news.json")
    val_news_json_path = os.path.join(RAW_DIR, "valid_original_law.json")

[[ local ]]


In [3]:
# 기타 환경 설정
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import gc


# 시각화 관련 설정
try:
    plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
except:
    try:
        plt.rcParams['font.family'] = 'NanumGothic'
    except:
        plt.rcParams['font.family'] = 'AppleGothic'

plt.rcParams['axes.unicode_minus'] = False
fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)

Matplotlib is building the font cache; this may take a moment.


In [4]:
import json

with open(train_edit_json_path, "r", encoding="utf-8") as f:
    raw_train_edit = json.load(f)

In [58]:
data = raw_train_edit["documents"][0]["text"]

data

[[{'index': 0,
   'sentence': '이명박 대통령이 어제 30대 그룹 총수를 모아놓고 "시대적 요구는 역시 총수가 앞장서야 한다. 이미 상당한 변화의 조짐이 있다는 것을 고맙게 생각한다. 총수들께서 직접 관심을 가져주시면 빨리 전파돼 긍정적인 평가를 받을 수 있다고 본다"고 말했다.',
   'highlight_indices': '9,11;37,39;53,55;91,93;104,106'},
  {'index': 1,
   'sentence': "언뜻 보아 무슨 말인지 불분명하나 이 대통령이 지난 8ㆍ15 연설 후 정몽준 의원, 정몽구 현대차 회장이 각각 2000억원과 5000억원을 기부한 사실과 '공생발전'이란 화두를 연결하면 금방 짐작이 간다.",
   'highlight_indices': '0,2;6,8;14,16;59,61;104,106'},
  {'index': 2,
   'sentence': '다른 그룹 총수들도 좀 나서라고 은근히 떠민 것이다.',
   'highlight_indices': '0,2;11,12;18,21'},
  {'index': 3,
   'sentence': '이 대통령은 기부에 대한 후속 선언이 나오지 않은 탓인지 총수들의 사회공헌 방안에 불만을 표시했다는 후문이다.',
   'highlight_indices': '0,1'}],
 [{'index': 4,
   'sentence': '최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의 자산세를 거둬 약 155조원을 마련하자는 논의가 있었다.',
   'highlight_indices': '55,56'},
  {'index': 5,
   'sentence': '이런 흐름에 한국만 동떨어져 있기는 어려운 게 글로벌 시대의 특징이다.',
   'highlight_indices': '0,2'},
  {'index': 6,
   'sentence': '항간에는 이번 회동 후 삼성을 비롯해 몇몇 그룹이 노블레스 오블리주 방안을 준비하고 있

In [51]:
for dict_ in data:
    sentence = dict_["sentence"]
    highlight_indices = dict_["highlight_indices"].split(";")
    indices = [(int(a.split(",")[0]), int(a.split(",")[1])) for a in highlight_indices]

    for i in range(len(indices)):
        a = indices[-(i+1)]
        sentence = sentence[:a[1]] + "</A>" + sentence[a[1]:]
        sentence = sentence[:a[0]] + "<A>" + sentence[a[0]:]

    print(sentence)

이명박 대통령이 <A>어제</A> 30대 그룹 총수를 모아놓고 "시대적 요구는 <A>역시</A> 총수가 앞장서야 한다. <A>이미</A> 상당한 변화의 조짐이 있다는 것을 고맙게 생각한다. 총수들께서 <A>직접</A> 관심을 가져주시면 <A>빨리</A> 전파돼 긍정적인 평가를 받을 수 있다고 본다"고 말했다.
<A>언뜻</A> 보아 <A>무슨</A> 말인지 불<A>분명</A>하나 이 대통령이 지난 8ㆍ15 연설 후 정몽준 의원, 정몽구 현대차 회장이 <A>각각</A> 2000억원과 5000억원을 기부한 사실과 '공생발전'이란 화두를 연결하면 <A>금방</A> 짐작이 간다.
<A>다른</A> 그룹 총수들도 <A>좀</A> 나서라고 <A>은근히</A> 떠민 것이다.
<A>이</A> 대통령은 기부에 대한 후속 선언이 나오지 않은 탓인지 총수들의 사회공헌 방안에 불만을 표시했다는 후문이다.


In [70]:
def get_text(article):
    final_sentence = ""

    for paragraph in article:

        for sentence_dict in paragraph:
            sentence = sentence_dict["sentence"]
            highlight_indices = sentence_dict["highlight_indices"].split(";")
            if highlight_indices != [""]:
                indices = [(int(a.split(",")[0]), int(a.split(",")[1])) for a in highlight_indices]
                for i in range(len(indices)):
                    a = indices[-(i+1)]
                    sentence = sentence[:a[1]] + "</A>" + sentence[a[1]:]
                    sentence = sentence[:a[0]] + "<A>" + sentence[a[0]:]

            final_sentence += " " + sentence
            final_sentence = final_sentence.lstrip()

    return final_sentence

article = raw_train_edit["documents"][3]["text"]
get_text(article)

'이명박 대통령이 <A>어제</A> 30대 그룹 총수를 모아놓고 "시대적 요구는 <A>역시</A> 총수가 앞장서야 한다. <A>이미</A> 상당한 변화의 조짐이 있다는 것을 고맙게 생각한다. 총수들께서 <A>직접</A> 관심을 가져주시면 <A>빨리</A> 전파돼 긍정적인 평가를 받을 수 있다고 본다"고 말했다. <A>언뜻</A> 보아 <A>무슨</A> 말인지 불<A>분명</A>하나 이 대통령이 지난 8ㆍ15 연설 후 정몽준 의원, 정몽구 현대차 회장이 <A>각각</A> 2000억원과 5000억원을 기부한 사실과 \'공생발전\'이란 화두를 연결하면 <A>금방</A> 짐작이 간다. <A>다른</A> 그룹 총수들도 <A>좀</A> 나서라고 <A>은근히</A> 떠민 것이다. <A>이</A> 대통령은 기부에 대한 후속 선언이 나오지 않은 탓인지 총수들의 사회공헌 방안에 불만을 표시했다는 후문이다. 최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의 자산세를 거둬 <A>약</A> 155조원을 마련하자는 논의가 있었다. <A>이런</A> 흐름에 한국만 동떨어져 있기는 어려운 게 글로벌 시대의 특징이다. 항간에는 이번 회동 후 삼성을 비롯해 <A>몇몇</A> 그룹이 노블레스 오블리주 방안을 준비하고 있다는 말이 나도는데 대통령의 강요나 포퓰리즘에 의한 압박보다 자발적 문화로 만들어가야 효과가 큰 법이다. <A>그런</A> 면에서 재계에 적절한 방안 마련을 맡기고 정치권이나 여론은 <A>너무</A> 압박하지 말고 시간을 줘야 한다. 국가채무 문제로 글로벌 경기 침체 우려가 큰 상황에서 기업들은 \'생존\'에 큰 부담을 느끼고 있기 때문이다. 이날 전경련에 따르면 30대 그룹은 올해 고용 12만4000명, 투자 114조원 등 \'선물\'을 준비했다. 세계적인 더블딥이 우려되는 상황에서 공격경영이 어렵겠지만 연초 <A>한</A> 번 발표한 내용을 <A>약간</A> 수정해 내놓은 전경련의 행태는 답답하다. 설립 50주년이 됐으면 <A>좀</A> <A

In [7]:
from collections.abc import Mapping, Sequence
from pathlib import Path

def dig(data, path="root", level=0, seen=None):
    if seen is None:
        seen = set()
    indent = "  " * level
    if isinstance(data, (Mapping, Sequence)) and not isinstance(data, (str, bytes, bytearray)):
        obj_id = id(data)
        if obj_id in seen:
            print(f"{indent}{path}: <cycle>")
            return
        seen.add(obj_id)

    if isinstance(data, Mapping):
        print(f"{indent}{path}: dict ({len(data)})")
        for key, value in data.items():
            next_path = f"{path}.{key}" if path else str(key)
            dig(value, next_path, level + 1, seen)
    elif isinstance(data, Sequence) and not isinstance(data, (str, bytes, bytearray)):
        print(f"{indent}{path}: list ({len(data)})")
        if data:
            dig(data[0], f"{path}[0]", level + 1, seen)
        else:
            print(f"{indent}  {path}[empty]: -> empty")
    else:
        print(f"{indent}{path}: -> {type(data).__name__}")

raw_train_edit = json.loads(Path(train_edit_json_path).read_text())
dig(raw_train_edit)


root: dict (3)
  root.name: -> str
  root.delivery_date: -> str
  root.documents: list (56760)
    root.documents[0]: dict (14)
      root.documents[0].id: -> str
      root.documents[0].category: -> str
      root.documents[0].media_type: -> str
      root.documents[0].media_sub_type: -> str
      root.documents[0].media_name: -> str
      root.documents[0].size: -> str
      root.documents[0].char_count: -> str
      root.documents[0].publish_date: -> str
      root.documents[0].title: -> str
      root.documents[0].text: list (5)
        root.documents[0].text[0]: list (4)
          root.documents[0].text[0][0]: dict (3)
            root.documents[0].text[0][0].index: -> int
            root.documents[0].text[0][0].sentence: -> str
            root.documents[0].text[0][0].highlight_indices: -> str
      root.documents[0].annotator_id: -> int
      root.documents[0].document_quality_scores: dict (4)
        root.documents[0].document_quality_scores.readable: -> int
        root.docum

In [27]:
from collections.abc import Mapping, Sequence
import pandas as pd

def iter_leaf_paths(node, prefix=()):
    if isinstance(node, Mapping):
        if not node:
            yield prefix, None
        else:
            for key, value in node.items():
                yield from iter_leaf_paths(value, prefix + (key,))
    elif isinstance(node, Sequence) and not isinstance(node, (str, bytes, bytearray)):
        if not node:
            yield prefix + ('__empty__',), None
        else:
            for idx, value in enumerate(node):
                yield from iter_leaf_paths(value, prefix + (idx,))
    else:
        yield prefix, node

def flatten_document(document, joiner=".", keep_keys=None):
    keep_keys = keep_keys or set()
    flattened = {}
    for key in keep_keys:
        if key in document:
            flattened[key] = document[key]  # text 등은 통째로 저장
    for path, value in iter_leaf_paths(document):
        if path and path[0] in keep_keys:
            continue  # text.* 경로는 펼치지 않음
        flattened[joiner.join(map(str, path))] = value
    return flattened

KEEP_AS_IS = {"text"}
train_edit_meta = {f"dataset.{k}": v for k, v in raw_train_edit.items() if k != "documents"}
train_edit_rows = [
    {**train_edit_meta, **flatten_document(doc, keep_keys=KEEP_AS_IS)}
    for doc in raw_train_edit["documents"]
    ]


train_edit_df = pd.DataFrame(train_edit_rows)

train_edit_df.columns

Index(['dataset.name', 'dataset.delivery_date', 'text', 'id', 'category',
       'media_type', 'media_sub_type', 'media_name', 'size', 'char_count',
       'publish_date', 'title', 'annotator_id',
       'document_quality_scores.readable', 'document_quality_scores.accurate',
       'document_quality_scores.informative',
       'document_quality_scores.trustworthy', 'extractive.0', 'extractive.1',
       'extractive.2', 'abstractive.0', 'drop_char_count'],
      dtype='object')

In [10]:
train_edit_df.head(1)

,dataset.name,dataset.delivery_date,text,id,category,media_type,media_sub_type,media_name,size,char_count,...,annotator_id,document_quality_scores.readable,document_quality_scores.accurate,document_quality_scores.informative,document_quality_scores.trustworthy,extractive.0,extractive.1,extractive.2,abstractive.0,drop_char_count
0,사설/잡지 문서 프로젝트,2020-12-23 15:00:19,"[[{'index': 0, 'sentence': '이명박 대통령이 어제 30대 그룹...",100062073,오피니언,online,경제지,매일경제,medium,1153,...,3924,4,3,3,3,0,6,7,이명박 대통령은 어제 30대 그룹 총수를 모아놓고 시대적 요구는 역시 총수가 앞장서...,NaN


In [11]:
column_list = [column for column in train_edit_df.columns if column != "text"]

sum(train_edit_df[column_list].duplicated())

0

In [12]:
train_edit_df.iloc[1]["text"][0]

[{'index': 0,
  'sentence': '이명박 정부의 첫 대통령실장을 지낸 류우익 주중대사가 통일부 장관에 내정되면서 대북 정책에 변화가 예상된다.',
  'highlight_indices': '8,9'},
 {'index': 1,
  'sentence': '청와대는 "지금까지의 통일 정책 일관성을 유지해갈 것"이라고 강조했지만, 현인택 장관 경질은 해임건의안을 제출한 야당 측 요구를 의식한 측면이 크다.',
  'highlight_indices': ''},
 {'index': 2,
  'sentence': '이 대통령의 인사 스타일에 비춰보더라도 기존 정책 기조 유지가 목표라면 2년7개월째 자리를 지킨 현 장관을 교체할 이유가 없었을 것이다.',
  'highlight_indices': '0,1;54,55'}]

In [ ]:
from collections.abc import Mapping, Sequence
from pathlib import Path
import json

def iter_leaves(node, prefix=()):
    if isinstance(node, Mapping):
        if not node:
            yield prefix, None
        else:
            for key, value in node.items():
                yield from iter_leaves(value, prefix + (key,))
    elif isinstance(node, Sequence) and not isinstance(node, (str, bytes, bytearray)):
        if not node:
            yield prefix + ('__empty__',), None
        else:
            for idx, value in enumerate(node):
                yield from iter_leaves(value, prefix + (idx,))
    else:
        yield prefix, node

def flatten(node):
    return {".".join(map(str, path)): value for path, value in iter_leaves(node)}

raw = json.loads(Path(train_edit_json_path).read_text())
meta = {f"dataset.{k}": v for k, v in raw.items() if k != "documents"}
rows = [{**meta, **flatten(doc)} for doc in raw["documents"]]

In [65]:
import pandas as pd

train_edit_df = pd.DataFrame(rows)
dup_dict = dict()

for i in range(1, 45):
    column  = f"text.{i}.0.sentence"
    mask = train_edit_df[column].notna()
    dupes = train_edit_df.loc[mask, column].duplicated(keep=False)
    result = train_edit_df[mask & dupes]

    if len(result) > 0:
        print(f"i: {i} ({len(result)}개)")
        print(result[column])
        dup_dict[i] = {result[column].index: result[column]}

i: 1 (7876개)
0        최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의...
3        최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의...
5        물가가 이렇게 뜀박질한 것은 2008년 8월(5.6%) 이래 36개월 만인데 당시는...
6        본회의 표결을 지켜 보려 참석했던 방청객과 기자들은 박희태 의장의 개의선언과 함께 ...
7                  그러나 법원 결정에도 불구하고 반대단체들은 결코 물러설 기세가 아니다.
                               ...                        
56714       우리 경제는 수출이 성장에 미치는 비중이 70% 이상으로, 수출로 먹고사는 구조다.
56715    일본 정부는 그동안 이 품목들의 한국 수출 절차를 간소화하는 우대 조치를 취해왔으나...
56751       우리 경제는 수출이 성장에 미치는 비중이 70% 이상으로, 수출로 먹고사는 구조다.
56752    일본 정부는 그동안 이 품목들의 한국 수출 절차를 간소화하는 우대 조치를 취해왔으나...
56753    우선 여야는 현행 3 개월인 탄력근로제 단위 기간을 확대하는 내용을 담은 관련 법 ...
Name: text.1.0.sentence, Length: 7876, dtype: object


TypeError: unhashable type: 'Index'

In [ ]:
dup_dict = {}

for i in range(1, 45):
    column = f"text.{i}.0.sentence"
    mask = train_edit_df[column].notna()
    dupes = train_edit_df.loc[mask, column].duplicated(keep=False)
    result = train_edit_df[mask & dupes]

    if not result.empty:
        print(f"i: {i} ({len(result)}개)")
        print(result[column])
        dup_dict[i] = result[column].to_dict()  # {row_index: sentence}

i: 1 (7876개)
0        최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의...
3        최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의...
5        물가가 이렇게 뜀박질한 것은 2008년 8월(5.6%) 이래 36개월 만인데 당시는...
6        본회의 표결을 지켜 보려 참석했던 방청객과 기자들은 박희태 의장의 개의선언과 함께 ...
7                  그러나 법원 결정에도 불구하고 반대단체들은 결코 물러설 기세가 아니다.
                               ...                        
56714       우리 경제는 수출이 성장에 미치는 비중이 70% 이상으로, 수출로 먹고사는 구조다.
56715    일본 정부는 그동안 이 품목들의 한국 수출 절차를 간소화하는 우대 조치를 취해왔으나...
56751       우리 경제는 수출이 성장에 미치는 비중이 70% 이상으로, 수출로 먹고사는 구조다.
56752    일본 정부는 그동안 이 품목들의 한국 수출 절차를 간소화하는 우대 조치를 취해왔으나...
56753    우선 여야는 현행 3 개월인 탄력근로제 단위 기간을 확대하는 내용을 담은 관련 법 ...
Name: text.1.0.sentence, Length: 7876, dtype: object
i: 2 (7904개)
0        그런 면에서 재계에 적절한 방안 마련을 맡기고 정치권이나 여론은 너무 압박하지 말고...
3        그런 면에서 재계에 적절한 방안 마련을 맡기고 정치권이나 여론은 너무 압박하지 말고...
5        지난달 물가는 집중호우 등에 따른 농축수산물 가격 급등(13.3%)에 큰 타격을 입었다.
6        더구나 표결에 앞서 작년까지 국회의장을 지낸 김형오 의원은 "죄 없는 사람이 이 여...
7        3일에는 서울에서 '평화비행기'를 타고 강정마을을

In [35]:
len(dup_dict)

17

In [ ]:
dup_dict[1]

dupdup_dict = {}

for key_num in dup_dict.keys():
    for key, value in dup_dict[key_num].items():
        if value in dupdup_dict.keys():
            dupdup_dict[value].add(key)
        else:
            dupdup_dict[value] = {key}

dupdup_dict

for key, value in dupdup_dict.items():
    

{'최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의 자산세를 거둬 약 155조원을 마련하자는 논의가 있었다.': {0,
  3},
 '물가가 이렇게 뜀박질한 것은 2008년 8월(5.6%) 이래 36개월 만인데 당시는 원유 등 국제 원자재값 급등 때문이었다.': {5,
  8},
 '본회의 표결을 지켜 보려 참석했던 방청객과 기자들은 박희태 의장의 개의선언과 함께 밖으로 내몰렸다.': {6, 9},
 '그러나 법원 결정에도 불구하고 반대단체들은 결코 물러설 기세가 아니다.': {7, 10},
 '또한 대법원 전원합의체(양창수 대법관)는 이날 광우병 보도에 대한 정정ㆍ반론보도 청구소송에 대해 우리 국민이 광우병에 걸릴 위험이 크다는 내용만 정정하면 된다고 판결했다.': {11,
  14},
 '공정위는 유통업체들이 판매수수료를 낮춰 입점업체들에 1000억원의 혜택이 돌아가도록 주문했다고 한다.': {12, 15},
 '국세청은 국외 부동산 취득이나 고액 외환거래 사실이 있는 이들을 비롯해 2000여 명에게 신고 안내문을 보냈지만 실제 신고한 이는 211명에 불과했다.': {16,
  19},
 '고용은 경기 상황이 일정한 시차를 두고 영향을 미치는 경기후행적인 지표다.': {17, 20},
 "다만 우리가 주목하는 것은 '안철수 돌풍'이 기성 정치권에 던지는 경고 메시지다.": {18, 21},
 '다른 전문직들도 거의 비슷하지만 연예인 탈세는 대개 두 가지 유형으로 나타난다.': {22, 24},
 '권혁세 금융감독원장은 "이번 경영 진단은 어느 때보다 엄정하게 실시됐다"며 "충분한 자구 노력을 유도해 기준을 맞추면 정상화 길로 유도하고 그렇지 못하면 영업정지 조치를 취할 것"이라고 밝혔다.': {23,
  25},
 '안 교수는 이제부터 백신프로그램을 무료로 제공해온 착한 기업인이나 고결한 성품을 가진 학자가 아니라 현실 정치인으로 신분이 바뀐다.': {27,
  29},
 '때마침 경찰도 불법 집회와 시위에 대한 대응 수

In [30]:
for value in dup_dict.values():
    for index, sentence in value.items():
        print(index)
        print(sentence)
        break

0
최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의 자산세를 거둬 약 155조원을 마련하자는 논의가 있었다.
0
그런 면에서 재계에 적절한 방안 마련을 맡기고 정치권이나 여론은 너무 압박하지 말고 시간을 줘야 한다.
0
이날 전경련에 따르면 30대 그룹은 올해 고용 12만4000명, 투자 114조원 등 '선물'을 준비했다.
0
허창수 전경련 회장은 "대기업ㆍ중소기업이 서로 공생하고 발전할 수 있도록 노력하겠다. 기업이 사회적 책임을 다하겠다"는 원론적인 발언에 그쳐 전경련 특유의 무미건조함을 드러냈다.
5
지식경제부는 8월에 휴가가 몰려 있어 나타난 계절적 요인을 들고 있지만, 아무리 그렇다 하더라도 감소폭은 심각하다.
5
무역흑자가 그동안 한국 경제를 이끄는 견인차 역할을 해 온 점을 감안하면 수지 악화는 큰 걱정이다.
5
정부는 앞으로 나아질 것이라는 막연한 말보다 상황이 얼마나 어려운지 소상히 설명하고 할 수 있는 근본적인 대책 마련에 최선을 다하기 바란다.
811
현재 부동산시장은 거래가 늘지 않고 가격도 오르지 않는 심각한 지경에 빠져 있다.
1805
애플 진영은 삼성 측 표준특허를 무력화하기 위해 유럽시장에서 대대적인 공세에 나설 텐데 삼성은 이 전쟁에서 이겨야 한다.
2926
북한은 공단 폐쇄와 같은 어리석은 짓은 삼가고 대화의 장을 열어놓고 주변국과 유연한 해법을 공동 모색하기 바란다.
10905
<글·윤무영 그림·김용민>
12817
<글·윤무영 그림·김용민>
9922
<글·윤무영 그림·김용민>
10957
<글·윤무영 그림·김용민>
9323
<글·윤무영 그림·김용민>
42170
농식품부 관계자는 "실태조사를 한 뒤 각계의 의견을 수렴할 예정"이라며 "현재 유기동물 구조 업무는 기본적으로 지자체 사무로 돼 있지만, 사설동물보호소 관리·감독 업무는 어디에 맡길지 고민해봐야 할 것"이라고 덧붙였다.
9855
<글·윤무영 그림·김용민>
